# Feature Engineering for anime users

## Imports and Set Ups

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


New Code without crashing RAM

In [ ]:
import pandas as pd
import numpy as np
import json
import gc # Garbage Collector
from sklearn.model_selection import train_test_split

input_folder_path = '/content/drive/MyDrive/yuran_files/datasets_needed'
output_folder_path = '/content/drive/MyDrive/yuran_files'

print("1. Loading Data with Memory Optimization...")
# Downcast data types to save massive amounts of RAM
dtypes = {
    'user_id': np.int32,
    'anime_id': np.int32,
    'rating': np.float32,
    'watching_status': np.int8,
    'watched_episodes': np.int32
}

# Load interactions (only what we need)
animelist = pd.read_csv(f'{input_folder_path}/animelist_final_sampled.csv', dtype=dtypes)

# Load anime master (only MAL_ID and Episodes)
anime_comp = pd.read_csv(f'{input_folder_path}/anime_complete.csv', usecols=['MAL_ID', 'Episodes'])
anime_comp.rename(columns={'MAL_ID': 'anime_id'}, inplace=True)

# Clean Episodes directly to numbers
anime_comp['Episodes'] = pd.to_numeric(anime_comp['Episodes'], errors='coerce').fillna(0).astype(np.int32)

print("2. Merging Datasets...")
animelist = animelist.merge(anime_comp, how='left', on='anime_id')
del anime_comp # FREE MEMORY
gc.collect()

print("3. Feature Engineering (watched_episode_ratio)...")
# Calculate ratio, filling divisions by zero with 0
animelist['watched_episode_ratio'] = (animelist['watched_episodes'] / animelist['Episodes'].replace(0, np.nan)).fillna(0).astype(np.float32)

print("4. Applying ID Mappings...")
with open(f'{input_folder_path}/user_id_map.json', 'r') as f:
    user_mapping = json.load(f)
with open(f'{input_folder_path}/anime_id_map.json', 'r') as f:
    anime_mapping = json.load(f)

# Convert JSON string keys to integers
user_mapping = {int(k): int(v) for k, v in user_mapping.items()}
anime_mapping = {int(k): int(v) for k, v in anime_mapping.items()}

animelist['user_idx'] = animelist['user_id'].map(user_mapping)
animelist['anime_idx'] = animelist['anime_id'].map(anime_mapping)

del user_mapping, anime_mapping # FREE MEMORY
gc.collect()

# Clean up mapped rows
animelist = animelist.dropna(subset=['user_idx', 'anime_idx'])
animelist['user_idx'] = animelist['user_idx'].astype(np.int32)
animelist['anime_idx'] = animelist['anime_idx'].astype(np.int32)

print("5. Splitting Data (Train/Val/Test)...")
# Split 1: 80% Train, 20% Temp
train_df, temp_df = train_test_split(animelist, test_size=0.2, random_state=42)
del animelist # FREE MASSIVE DATAFRAME BEFORE NEXT SPLIT
gc.collect()

# Split 2: 10% Val, 10% Test
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
del temp_df # FREE MEMORY
gc.collect()

print("6. Saving final files to Google Drive...")
train_df.to_csv(f'{output_folder_path}/train_complete.csv', index=False)
val_df.to_csv(f'{output_folder_path}/val_complete.csv', index=False)
test_df.to_csv(f'{output_folder_path}/test_complete.csv', index=False)

print("✅ Success! Memory-Optimized Pipeline Complete!")

1. Loading Data with Memory Optimization...
2. Merging Datasets...
3. Feature Engineering (watched_episode_ratio)...
4. Applying ID Mappings...
5. Splitting Data (Train/Val/Test)...
6. Saving final files to Google Drive...
✅ Success! Memory-Optimized Pipeline Complete!


In [ ]:
# Import libraries
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_columns', None)

## Data Profiling of animelist.csv

In [ ]:
input_folder_path = '/content/drive/MyDrive/yuran_files/datasets_needed'
animelist = pd.read_csv(f'{input_folder_path}/animelist.csv')

print(f"Shape of animelist: {animelist.shape}")
print("animelist:")
animelist.head()

Shape of animelist: (109224747, 5)
animelist:


,user_id,anime_id,rating,watching_status,watched_episodes
0,0,67,9,1,1
1,0,6702,7,1,4
2,0,242,10,1,4
3,0,4898,0,1,1
4,0,21,10,1,0


In [ ]:
print("animelist describe()")
animelist.describe(include='all')

animelist describe()


,user_id,anime_id,rating,watching_status,watched_episodes
count,1.092247e+08,1.092247e+08,1.092247e+08,1.092247e+08,1.092247e+08
mean,1.768098e+05,1.649590e+04,4.245717e+00,3.087289e+00,1.210818e+01
std,1.018487e+05,1.379737e+04,3.912888e+00,1.774407e+00,1.463155e+02
min,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,8.849100e+04,3.194000e+03,0.000000e+00,2.000000e+00,0.000000e+00
50%,1.771420e+05,1.244500e+04,5.000000e+00,2.000000e+00,3.000000e+00
75%,2.651870e+05,3.083100e+04,8.000000e+00,6.000000e+00,1.200000e+01
max,3.534040e+05,4.849200e+04,1.000000e+01,5.500000e+01,6.553500e+04


In [ ]:
print("animelist info()")
animelist.info()

animelist info()
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109224747 entries, 0 to 109224746
Data columns (total 5 columns):
 #   Column            Dtype
---  ------            -----
 0   user_id           int64
 1   anime_id          int64
 2   rating            int64
 3   watching_status   int64
 4   watched_episodes  int64
dtypes: int64(5)
memory usage: 4.1 GB


In [ ]:
print("Missing values")
print(animelist.isnull().sum())

Missing values
user_id             0
anime_id            0
rating              0
watching_status     0
watched_episodes    0
dtype: int64


## Cleaning of animelist.csv

In [ ]:
# Remove invalid watch status from animelist
animelist_clean = animelist[
    (animelist["watching_status"] == 1) |
    (animelist["watching_status"] == 2) |
    (animelist["watching_status"] == 3) |
    (animelist["watching_status"] == 4) |
    (animelist["watching_status"] == 6)
]

print("animelist_clean:")
animelist_clean.head()

animelist_clean:


,user_id,anime_id,rating,watching_status,watched_episodes
0,0,67,9,1,1
1,0,6702,7,1,4
2,0,242,10,1,4
3,0,4898,0,1,1
4,0,21,10,1,0


## One Hot Encoding of Watch Status

In [ ]:
input_folder_path = '/content/drive/MyDrive/yuran_files/datasets_needed'
# Access watching_status.csv
watching_status = pd.read_csv(f'{input_folder_path}/watching_status.csv')

print("watching_status:")
watching_status

watching_status:


,status,description
0,1,Currently Watching
1,2,Completed
2,3,On Hold
3,4,Dropped
4,6,Plan to Watch


In [ ]:
# Rename watching_status to status
animelist_clean.rename(columns={'watching_status': 'status'}, inplace=True)

# Merge animelist_final_sampled with watching_status
animelist_with_watch_status_name = pd.merge(animelist_clean, watching_status, how="left", on="status")

animelist_with_watch_status_name = animelist_with_watch_status_name.drop(columns=['status'])

animelist_with_watch_status_name = animelist_with_watch_status_name.rename(columns={' description': 'watch_status'})

animelist_with_watch_status_name.head()

/tmp/ipykernel_13192/3547404828.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  animelist_clean.rename(columns={'watching_status': 'status'}, inplace=True)


,user_id,anime_id,rating,watched_episodes,watch_status
0,0,67,9,1,Currently Watching
1,0,6702,7,4,Currently Watching
2,0,242,10,4,Currently Watching
3,0,4898,0,1,Currently Watching
4,0,21,10,0,Currently Watching


In [ ]:
# OneHotEncoder for watch_status
watch_encoder = OneHotEncoder(handle_unknown='ignore')

encoded_watch_status = watch_encoder.fit_transform(
    animelist_with_watch_status_name[["watch_status"]]
)

one_hot_watch_status_df = pd.DataFrame(
    encoded_watch_status.toarray(),
    columns=watch_encoder.get_feature_names_out(["watch_status"])
)

animelist_encoded = pd.concat(
    [animelist_with_watch_status_name, one_hot_watch_status_df],
    axis=1
)

animelist_encoded = animelist_encoded.drop("watch_status", axis=1)

animelist_encoded.head()

,user_id,anime_id,rating,watched_episodes,watch_status_Completed,watch_status_Currently Watching,watch_status_Dropped,watch_status_On Hold,watch_status_Plan to Watch
0,0,67,9,1,0.0,1.0,0.0,0.0,0.0
1,0,6702,7,4,0.0,1.0,0.0,0.0,0.0
2,0,242,10,4,0.0,1.0,0.0,0.0,0.0
3,0,4898,0,1,0.0,1.0,0.0,0.0,0.0
4,0,21,10,0,0.0,1.0,0.0,0.0,0.0


## Add score_no_negative column

In [ ]:
animelist_encoded["rating_no_negative"] = animelist_encoded["rating"].apply(
    lambda x: None if x == 0 else x
)

print("View animelist_encoded after adding rating_no_negative:")
animelist_encoded.head()

View animelist_encoded after adding rating_no_negative:


,user_id,anime_id,rating,watched_episodes,watch_status_Completed,watch_status_Currently Watching,watch_status_Dropped,watch_status_On Hold,watch_status_Plan to Watch,rating_no_negative
0,0,67,9,1,0.0,1.0,0.0,0.0,0.0,9.0
1,0,6702,7,4,0.0,1.0,0.0,0.0,0.0,7.0
2,0,242,10,4,0.0,1.0,0.0,0.0,0.0,10.0
3,0,4898,0,1,0.0,1.0,0.0,0.0,0.0,NaN
4,0,21,10,0,0.0,1.0,0.0,0.0,0.0,10.0


## Add watched episodes ratio

In [ ]:
input_folder_path = '/content/drive/MyDrive/yuran_files/datasets_needed'
# Access anime_complete.csv
anime_complete = pd.read_csv(f'{input_folder_path}/anime_complete.csv')

# Rename MAL_ID to anime_id
anime_complete.rename(columns={'MAL_ID': 'anime_id'}, inplace=True)

# Select required columns from anime_complete
selected_anime_complete = anime_complete[['anime_id', 'Episodes']]

selected_anime_complete.head()

,anime_id,Episodes
0,1,26
1,5,1
2,6,26
3,7,26
4,8,52


In [ ]:
# Convert Episodes to numeric
def normalize_episode_count(ep):
    if pd.isna(ep):
        return 0
    if isinstance(ep, str):
        ep = ep.strip()
        if ep == "" or ep.lower() == "unknown":
            return 0
    try:
        ep = int(float(ep))
        return max(ep, 0)
    except:
        return 0
selected_anime_complete['Episodes'] = selected_anime_complete["Episodes"].apply(normalize_episode_count)

/tmp/ipykernel_13192/2415297018.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_anime_complete['Episodes'] = selected_anime_complete["Episodes"].apply(normalize_episode_count)


In [ ]:
selected_anime_complete.head()

,anime_id,Episodes
0,1,26
1,5,1
2,6,26
3,7,26
4,8,52


In [ ]:
# Merge anime_complete with animelist_encoded
animelist_anime = pd.merge(animelist_encoded, selected_anime_complete, how="left", on="anime_id")

# Create watched_episodes_ratio column
animelist_anime["watched_episode_ratio"] = animelist_anime["watched_episodes"] / animelist_anime["Episodes"]

# View animelist_anime
print("animelist_anime")
animelist_anime.head()

animelist_anime


,user_id,anime_id,rating,watched_episodes,watch_status_Completed,watch_status_Currently Watching,watch_status_Dropped,watch_status_On Hold,watch_status_Plan to Watch,rating_no_negative,Episodes,watched_episode_ratio
0,0,67,9,1,0.0,1.0,0.0,0.0,0.0,9.0,24,0.041667
1,0,6702,7,4,0.0,1.0,0.0,0.0,0.0,7.0,175,0.022857
2,0,242,10,4,0.0,1.0,0.0,0.0,0.0,10.0,13,0.307692
3,0,4898,0,1,0.0,1.0,0.0,0.0,0.0,NaN,24,0.041667
4,0,21,10,0,0.0,1.0,0.0,0.0,0.0,10.0,0,NaN


## Save processed animelist_file

In [ ]:
import json
from sklearn.model_selection import train_test_split

input_folder_path = '/content/drive/MyDrive/yuran_files/datasets_needed'
output_folder_path = '/content/drive/MyDrive/yuran_files'

print("1. Loading your ID mappings...")
with open(f'{input_folder_path}/user_id_map.json', 'r') as f:
    user_mapping = json.load(f)
with open(f'{input_folder_path}/anime_id_map.json', 'r') as f:
    anime_mapping = json.load(f)

print("2. Applying mappings to Xin Wei's new feature dataset...")
# animelist_anime is the dataframe she created in her notebook
animelist_anime['user_idx'] = animelist_anime['user_id'].astype(str).map(user_mapping)
animelist_anime['anime_idx'] = animelist_anime['anime_id'].astype(str).map(anime_mapping)

# Drop any rows that didn't map correctly
animelist_anime = animelist_anime.dropna(subset=['user_idx', 'anime_idx'])
animelist_anime['user_idx'] = animelist_anime['user_idx'].astype(int)
animelist_anime['anime_idx'] = animelist_anime['anime_idx'].astype(int)

print("3. Re-Splitting into Train/Val/Test with the new features...")
train_df, temp_df = train_test_split(animelist_anime, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Overwrite your old splits with this new, feature-rich data!
train_df.to_csv(f'{output_folder_path}/train_complete.csv', index=False)
val_df.to_csv(f'{output_folder_path}/val_complete.csv', index=False)
test_df.to_csv(f'{output_folder_path}/test_complete.csv', index=False)

print("✅ Success! The Train/Val/Test datasets now include Xin Wei's new user features and are mapped perfectly.")

1. Loading your ID mappings...
2. Applying mappings to Xin Wei's new feature dataset...
3. Re-Splitting into Train/Val/Test with the new features...
✅ Success! The Train/Val/Test datasets now include Xin Wei's new user features and are mapped perfectly.
